# Final Project Step - Kyle

**Student Name:** Kyle Hagerman

**Group:** 2

**Student ID:** 400383386

**CodaBench Username:** hagorman

**Code Attrition:**

All code was copied from my Assignmnent 4, and repurposed for this task.

# Setup

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
# Core imports
import os
import random
import numpy as np
import pandas as pd

# HuggingFace
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Scikit-learn
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load Data

In [ ]:
base_path = '/content/drive/MyDrive/Uni/Year 4/Term 2/Natural Language Processing 4NL3/FinalProjectData'

try:
    # Load train and validation csvs directly
    train_dataset = load_dataset('csv', data_files=os.path.join(base_path, 'genre_train.csv'))['train']
    val_dataset = load_dataset('csv', data_files=os.path.join(base_path, 'genre_valid.csv'))['train']

    # Load test data separately
    test_lyrics_df = pd.read_csv(os.path.join(base_path, 'genre_test.csv'))
    test_labels_df = pd.read_csv(os.path.join(base_path, 'test_labels.csv'))

    # Merge on the common key 'lyric_id'
    test_df = pd.merge(test_lyrics_df, test_labels_df, on='lyric_id')
    test_dataset = Dataset.from_pandas(test_df)

    # Map string genres to integer labels (Pop = 0, Metal = 1)
    def map_labels(example):
        example['label'] = 0 if example['genre'].strip().lower() == 'p' else 1
        return example

    train_dataset = train_dataset.map(map_labels)
    val_dataset = val_dataset.map(map_labels)
    test_dataset = test_dataset.map(map_labels)

    print("Loaded all files. Example of training data:")
    print(train_dataset[0])
except Exception as e:
    print(f"Error loading files: {e}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30612 [00:00<?, ? examples/s]

Map:   0%|          | 0/3827 [00:00<?, ? examples/s]

Map:   0%|          | 0/3827 [00:00<?, ? examples/s]

Loaded all files. Example of training data:
{'lyric_id': 72800, 'genre': 'p', 'lyrics': "[Timbaland]\nThere's a lot of people out there\nThis the fricky, fricky under the track\nThere's a lot of people out there\nFricky, fricky, fricky, fricky, fricky, under they track, ow, tell 'em\n\n[Chorus: Timbaland]\nTell 'em it's on, I bet you didn't see it coming, oh\nTell 'em it's on, so what you gotta say now, oh\nTell 'em it's on, I bet you didn't see it coming, oh\nTell 'em it's on, so what you gotta say now, oh\nTell 'em it's on, V-A, G-A\nTell 'em it's on, so what you gotta say now, yo\nTell 'em it's on, yo, say what, say what\nTell 'em it's on\n\n[Pastor Troy]\nI'm the ?, the A-T-L with the flow\nKeep it coming, got the A-K under the doe\nWhy you runnin' talkin' mad but you ain't mad\nCause I'm bad, the P-T billy, the bad ass\nI heard you clappin' your jaw, talkin' bout the A-T-L\nHow you got it on lock, boy stop\nCause I'm reppin' the city, East Point to ?\nI come from the city that don

In [ ]:
# Explore the data
# Convert to pandas format temporarily for easy exploration
train_dataset.set_format(type="pandas")
df_explore = train_dataset[:]

print("First 5 Pop and 5 Metal songs:")
# Sample based on the string 'genre' column for readability
display(df_explore.groupby('genre').sample(n=5, random_state=42))

print("\nInfo about the training data:")
display(df_explore.info())

print("\nDistribution of each class:")
print(df_explore['genre'].value_counts())

# Reset format back to Arrow for Hugging Face processing
train_dataset.reset_format()

First 5 Pop and 5 Metal songs:


,lyric_id,genre,lyrics,label
23585,84716,m,LIVING SACRIFICE\nFeel your soul and let it tr...,1
20161,91528,m,Hear ye hear ye! The lady\nHelena has committe...,1
8811,101140,m,I've seen this too many times before Lust grip...,1
7025,12355,m,Well I can't stand to look at you now\nThis re...,1
24335,85706,m,"Many many years ago, when Persia came ashore\n...",1
16799,78754,p,"O come, all ye faithful\nJoyful and triumphant...",0
23433,96306,p,loving you isn't the right thing to do\nhow ca...,0
20547,90534,p,I said my tour just sold out\nAlbum's been bou...,0
1810,38456,p,"Just a little bit of love\nI was alone, I was ...",0
25653,50940,p,My girl's mad at me\nI didn't wanna go see the...,0



Info about the training data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30612 entries, 0 to 30611
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   lyric_id  30612 non-null  int64 
 1   genre     30612 non-null  object
 2   lyrics    30612 non-null  object
 3   label     30612 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 956.8+ KB


None


Distribution of each class:
genre
p    15306
m    15306
Name: count, dtype: int64


# Load Model

In [ ]:
MODEL_NAME = "google/electra-small-discriminator"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/54.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/54.2M [00:00<?, ?B/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Li

# Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    # max_length set to 512 (ELECTRA's maximum) since lyrics are long
    return tokenizer(examples['lyrics'], padding="max_length", truncation=True, max_length=512)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/30612 [00:00<?, ? examples/s]

Map:   0%|          | 0/3827 [00:00<?, ? examples/s]

Map:   0%|          | 0/3827 [00:00<?, ? examples/s]

# Train Model

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(base_path, "results_electra_music"),
    save_total_limit=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    greater_is_better=True,
    report_to=["none"],
)

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.from_numpy(logits).argmax(dim=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro"),
        "precision": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall": recall_score(labels, predictions, average="macro", zero_division=0)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.382351,0.350875,0.848707,0.848693,0.848824,0.848704
2,0.330040,0.339820,0.859943,0.859943,0.859943,0.859943
3,0.285222,0.319858,0.868304,0.868279,0.868578,0.868301
4,0.239858,0.355132,0.867259,0.867229,0.867595,0.867263
5,0.202150,0.365322,0.868565,0.868563,0.868589,0.868564


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=9570, training_loss=0.2971573282558716, metrics={'train_runtime': 1191.1151, 'train_samples_per_second': 128.501, 'train_steps_per_second': 8.034, 'total_flos': 4502970225745920.0, 'train_loss': 0.2971573282558716, 'epoch': 5.0})

# Evaluate on Test Set

In [ ]:
print("Predicting for test set...")
test_predictions = trainer.predict(tokenized_test)
binary_predictions = np.argmax(test_predictions.predictions, axis=-1)

# Ensure 'label' column is integer for ground truth comparison
y_true = np.array(tokenized_test['label'])
y_pred = binary_predictions

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary')

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"F1-Score: {f1 * 100:.2f}%\n")

print("Detailed Classification Report:")
# Updated target names to reflect our mapping
print(classification_report(y_true, y_pred, target_names=["Pop (0)", "Metal (1)"]))

Predicting for test set...


Accuracy: 86.73%
F1-Score: 86.65%

Detailed Classification Report:
              precision    recall  f1-score   support

     Pop (0)       0.86      0.87      0.87      1913
   Metal (1)       0.87      0.86      0.87      1914

    accuracy                           0.87      3827
   macro avg       0.87      0.87      0.87      3827
weighted avg       0.87      0.87      0.87      3827



In [ ]:
# Save predictions
test_results_df = pd.DataFrame({"label": binary_predictions})
test_results_df.to_csv(os.path.join(base_path, "test_predictions.csv"), index=None)

# END